# 0. Setup: install everything

Run this notebook once, before notebooks 1 to 6.

**What it does**
- Checks the Python version (3.11 or newer is needed).
- Makes sure the packages go into a **project environment**, never into your base Python or base Anaconda. If this notebook is already running in a virtual environment (for example the `.venv` made by `uv sync`) or a named conda environment, it uses that. Otherwise it creates `.venv/` in the repository folder.
- Installs every package the project uses, at the exact versions in `requirements.txt` (exported from `uv.lock`, the versions the results were produced with).
- Registers that environment as a Jupyter kernel called **Christmas Casuals (.venv)**, if it isn't the one running this notebook.
- On macOS, installs `libomp` (needed by xgboost) with Homebrew if it is missing.
- Checks that every package imports.

**Inputs**
- `../requirements.txt`

**Outputs**
- `../.venv/` (only if no project environment was running) and the Jupyter kernel above.

**Next:** if the last cell tells you to, switch the kernel of notebooks 1 to 6 to **Christmas Casuals (.venv)**. Then run them in order. Notebook 2 downloads the drifter data (about 8 GB) on its first run.

**If you use `uv`**, `uv sync` already does all of this; running this notebook just confirms everything is in place.

## 1. Python version

In [ ]:
import sys

print(f"Python {sys.version.split()[0]} at {sys.executable}")
assert sys.version_info >= (3, 11), "This project needs Python 3.11 or newer"

## 2. Choose the environment

A virtual environment (`sys.prefix` differs from `sys.base_prefix`) or a named conda environment (`.../envs/<name>`) counts as a project environment. Anything else, such as the system Python or base Anaconda, is left untouched and a new `.venv/` is created next to this repository's `pyproject.toml`.

In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

ROOT = Path("..").resolve()
prefix = Path(sys.prefix)
in_venv = sys.prefix != sys.base_prefix
in_conda_env = (prefix / "conda-meta").exists() and prefix.parent.name == "envs"

if in_venv or in_conda_env:
    python = sys.executable
    print(f"Using the environment this notebook runs in: {prefix}")
else:
    venv = ROOT / ".venv"
    if not venv.exists():
        print(f"{prefix} is a base Python, so creating a project environment at {venv}")
        subprocess.run([sys.executable, "-m", "venv", str(venv)], check=True)
    python = str(venv / ("Scripts/python.exe" if os.name == "nt" else "bin/python"))
    print(f"Installing into {python}")

## 3. Install the packages

The exact package versions come from `requirements.txt`, which is exported from `uv.lock`. They are installed with that environment's `pip`, or with `uv pip` if it has no `pip` (as in a `.venv` made by `uv`).

In [ ]:
requirements = ROOT / "requirements.txt"
print(f"Installing the pinned versions in {requirements.name}")

has_pip = subprocess.run([python, "-m", "pip", "--version"], capture_output=True).returncode == 0
if has_pip:
    cmd = [python, "-m", "pip", "install", "--quiet", "-r", str(requirements)]
elif shutil.which("uv"):
    cmd = ["uv", "pip", "install", "--quiet", "--python", python, "-r", str(requirements)]
else:
    raise RuntimeError("That environment has no pip and uv isn't installed; install uv or pip first")

subprocess.run(cmd, check=True)
print("Packages installed.")

if python != sys.executable:
    subprocess.run([python, "-m", "ipykernel", "install", "--user",
                    "--name", "christmas-casuals", "--display-name", "Christmas Casuals (.venv)"], check=True)
    print("Registered the Jupyter kernel 'Christmas Casuals (.venv)'.")

## 4. macOS only: OpenMP for xgboost

xgboost on macOS needs the `libomp` library. If xgboost can't load and Homebrew is available, this runs `brew install libomp`; otherwise it prints what to do.

In [ ]:
if sys.platform == "darwin":
    check = subprocess.run([python, "-c", "import xgboost"], capture_output=True, text=True)
    if check.returncode == 0:
        print("xgboost loads fine; libomp is already available.")
    elif shutil.which("brew"):
        subprocess.run(["brew", "install", "libomp"], check=True)
        print("Installed libomp.")
    else:
        print("xgboost needs libomp. Install Homebrew (https://brew.sh), then run `brew install libomp`.")
else:
    print("Not macOS; nothing to do.")

## 5. Check every package imports

This runs in the project environment chosen above, so it works even before you switch kernels.

In [ ]:
CHECK = '''
from importlib import import_module
from importlib.metadata import version
modules = {"numpy": "numpy", "pandas": "pandas", "scipy": "scipy", "sklearn": "scikit-learn",
           "statsmodels": "statsmodels", "xgboost": "xgboost", "matplotlib": "matplotlib",
           "openpyxl": "openpyxl", "clouddrift": "clouddrift", "cartopy": "cartopy"}
failed = []
for module, package in modules.items():
    try:
        import_module(module)
        print(f"  ok    {package} {version(package)}")
    except Exception as e:
        failed.append(package)
        print(f"  FAIL  {package}: {e}")
raise SystemExit(1 if failed else 0)
'''
result = subprocess.run([python, "-c", CHECK])
assert result.returncode == 0, "Some packages failed to import (see above)"

if python == sys.executable:
    print("\nEnvironment ready. Next: run notebooks 1 to 6 in order.")
else:
    print("\nEnvironment ready. Next: switch the kernel to 'Christmas Casuals (.venv)', then run notebooks 1 to 6 in order.")